# Arabic OCR Correction — All Phases Inference (Kaggle)

Run LLM inference for **all phases** on Kaggle GPU.

## Workflow
1. **Local** — Run `--mode export` for each phase to generate `inference_input.jsonl` files
2. **Upload** — Collect all exported files into a single Kaggle dataset
3. **Here** — Run this notebook (each cell runs one phase)
4. **Local** — Download corrections and run `--mode analyze` for each phase

## Expected Kaggle Dataset Structure
Upload your files preserving the `results/` directory structure:
```
your-kaggle-dataset/
├── phase2/inference_input.jsonl
├── phase3/inference_input.jsonl
├── phase4a/inference_input.jsonl
├── phase4b/inference_input.jsonl
├── phase4d/inference_input.jsonl
├── phase5/
│   ├── pair_conf_rules/inference_input.jsonl
│   ├── pair_conf_fewshot/inference_input.jsonl
│   ├── pair_rules_fewshot/inference_input.jsonl
│   ├── full_prompt/inference_input.jsonl
│   ├── abl_no_confusion/inference_input.jsonl
│   ├── abl_no_rules/inference_input.jsonl
│   ├── abl_no_fewshot/inference_input.jsonl
│   ├── self_reflective/inference_input.jsonl
│   ├── pair_self_conf/inference_input.jsonl
│   └── full_with_self/inference_input.jsonl
└── phase7/
    ├── inference_input.jsonl
    ├── dspy_trainset.jsonl
    └── dspy_devset.jsonl
```

## Before Running
- Set `KAGGLE_DATASET` (Cell 1) to your uploaded dataset name
- Set `REPO_URL` (Cell 2) to your GitHub repo URL
- Set `HF_REPO` (Cell 3) to your HuggingFace dataset repo
- Add `HF_TOKEN` as a Kaggle secret (Add-ons → Secrets)
- Enable **GPU T4 x2** accelerator and **Internet**

In [ ]:
# Cell 1 — Configuration
KAGGLE_DATASET = "YOUR_DATASET_NAME"  # <-- your uploaded Kaggle dataset name
REPO_URL = "https://github.com/YOUR_USERNAME/Arabic-Post-OCR-Correction.git"  # <-- your repo
HF_REPO = "YOUR_HF_USERNAME/arabic-ocr-corrections"  # <-- HuggingFace dataset repo
MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SYNC_EVERY = 100

# Derived paths
INPUT_ROOT = f"/kaggle/input/{KAGGLE_DATASET}"
OUTPUT_ROOT = "/kaggle/working/results"
PROJECT_DIR = "/kaggle/working/project"

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")

In [ ]:
# Cell 2 — Install dependencies & clone repo
!pip install transformers accelerate huggingface_hub pyyaml tqdm -q
# Uncomment for 4-bit quantization:
# !pip install bitsandbytes -q

!git clone {REPO_URL} {PROJECT_DIR}

In [ ]:
# Cell 3 — Helper: run infer.py for a given phase
import subprocess, sys, os

def run_inference(phase_name, input_subpath, output_subpath, hf_path=None):
    """Run scripts/infer.py for one phase."""
    input_file = f"{INPUT_ROOT}/{input_subpath}"
    output_file = f"{OUTPUT_ROOT}/{output_subpath}"

    # Create output directory
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if not os.path.exists(input_file):
        print(f"[SKIP] {phase_name}: input not found at {input_file}")
        return False

    print(f"\n{'='*60}")
    print(f"Running {phase_name}")
    print(f"  Input:  {input_file}")
    print(f"  Output: {output_file}")
    print(f"{'='*60}\n")

    cmd = [
        sys.executable, f"{PROJECT_DIR}/scripts/infer.py",
        "--input", input_file,
        "--output", output_file,
        "--model", MODEL,
        "--sync-every", str(SYNC_EVERY),
    ]
    if HF_REPO:
        cmd += ["--hf-repo", HF_REPO]
    if HF_TOKEN:
        cmd += ["--hf-token", HF_TOKEN]
    if hf_path:
        cmd += ["--hf-path", hf_path]

    result = subprocess.run(cmd)
    if result.returncode != 0:
        print(f"[ERROR] {phase_name} failed with return code {result.returncode}")
        return False
    print(f"[DONE] {phase_name}")
    return True

print("Helper ready.")

In [ ]:
# Cell 4 — Phase 2: Zero-Shot LLM
run_inference(
    "Phase 2 (Zero-Shot)",
    "phase2/inference_input.jsonl",
    "phase2/corrections.jsonl",
    hf_path="phase2/corrections.jsonl",
)

In [ ]:
# Cell 5 — Phase 3: OCR-Aware Prompting
run_inference(
    "Phase 3 (OCR-Aware)",
    "phase3/inference_input.jsonl",
    "phase3/corrections.jsonl",
    hf_path="phase3/corrections.jsonl",
)

In [ ]:
# Cell 6 — Phase 4A: Rule-Augmented
run_inference(
    "Phase 4A (Rule-Augmented)",
    "phase4a/inference_input.jsonl",
    "phase4a/corrections.jsonl",
    hf_path="phase4a/corrections.jsonl",
)

In [ ]:
# Cell 7 — Phase 4B: Few-Shot
run_inference(
    "Phase 4B (Few-Shot)",
    "phase4b/inference_input.jsonl",
    "phase4b/corrections.jsonl",
    hf_path="phase4b/corrections.jsonl",
)

In [ ]:
# Cell 8 — Phase 4D: Self-Reflective + Word Pairs
run_inference(
    "Phase 4D (Self-Reflective)",
    "phase4d/inference_input.jsonl",
    "phase4d/corrections.jsonl",
    hf_path="phase4d/corrections.jsonl",
)

In [ ]:
# Cell 9 — Phase 5: All Combinations
# Runs each combo sequentially. Skips combos whose input files are missing.

PHASE5_COMBOS = [
    "pair_conf_rules",
    "pair_conf_fewshot",
    "pair_rules_fewshot",
    "full_prompt",
    "abl_no_confusion",
    "abl_no_rules",
    "abl_no_fewshot",
    "self_reflective",
    "pair_self_conf",
    "full_with_self",
]

for combo in PHASE5_COMBOS:
    run_inference(
        f"Phase 5 / {combo}",
        f"phase5/{combo}/inference_input.jsonl",
        f"phase5/{combo}/corrections.jsonl",
        hf_path=f"phase5/{combo}/corrections.jsonl",
    )

In [ ]:
# Cell 10 — Phase 7: DSPy Prompt Optimization
# Phase 7 uses dspy_optimize.py (NOT infer.py) — it optimizes the prompt first, then infers.

!pip install dspy-ai -q

import subprocess, sys, os

trainset = f"{INPUT_ROOT}/phase7/dspy_trainset.jsonl"
devset   = f"{INPUT_ROOT}/phase7/dspy_devset.jsonl"
inp      = f"{INPUT_ROOT}/phase7/inference_input.jsonl"
out      = f"{OUTPUT_ROOT}/phase7/corrections.jsonl"

os.makedirs(os.path.dirname(out), exist_ok=True)

missing = [f for f in [trainset, devset, inp] if not os.path.exists(f)]
if missing:
    print(f"[SKIP] Phase 7: missing files: {missing}")
else:
    print(f"\n{'='*60}")
    print("Running Phase 7 (DSPy Optimization + Inference)")
    print(f"{'='*60}\n")

    cmd = [
        sys.executable, f"{PROJECT_DIR}/scripts/dspy_optimize.py",
        "--trainset", trainset,
        "--devset", devset,
        "--input", inp,
        "--output", out,
        "--model", MODEL,
    ]
    result = subprocess.run(cmd)
    if result.returncode != 0:
        print(f"[ERROR] Phase 7 failed with return code {result.returncode}")
    else:
        print("[DONE] Phase 7")

In [ ]:
# Cell 11 — Summary: list all output files
import os

print("\nGenerated correction files:")
print("=" * 60)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    for f in sorted(files):
        path = os.path.join(root, f)
        size_kb = os.path.getsize(path) / 1024
        rel = os.path.relpath(path, OUTPUT_ROOT)
        print(f"  {rel:50s} {size_kb:8.1f} KB")

print(f"\nDownload the '{OUTPUT_ROOT}' folder and place files under your local results/ directory.")
print("Then run --mode analyze for each phase locally.")